In [ ]:
import os
import shutil
from pathlib import Path

# ==========================================
# CONFIGURATION & PATH SETUP
# ==========================================
project_root = Path.cwd().parent

# Define the source directory (Your current multi-class N-RDD dataset)
# Change 'train' to 'val' later when you want to process the validation folder
split_name = 'n-rdd2024' 
source_dir = project_root / 'data' / split_name

source_images_dir = source_dir / 'images'
source_labels_dir = source_dir / 'labels'

# Define the target directory (The new, clean Single-Class dataset)
target_dir = project_root / 'data' / 'n-rdd2024_single_class'
target_images_dir = target_dir / 'images'
target_labels_dir = target_dir / 'labels'

# Create target directories if they don't exist
target_images_dir.mkdir(parents=True, exist_ok=True)
target_labels_dir.mkdir(parents=True, exist_ok=True)

# ==========================================
# LABEL MAPPING PARAMETERS
# ==========================================
# In N-RDD, 'D40' (Pothole) is usually index 4. 
# Double-check your old data.yaml if it's different.
OLD_POTHOLE_ID = '4'  

# We are converting this to a Single-Class problem, so the new ID must be 0
NEW_POTHOLE_ID = '0'  

print(f"[*] Starting Data Extraction for {split_name.upper()} split...")
print(f"[*] Filtering class ID '{OLD_POTHOLE_ID}' and converting to '{NEW_POTHOLE_ID}'...")

# ==========================================
# EXECUTION: FILTERING & COPYING
# ==========================================
all_label_files = list(source_labels_dir.glob('*.txt'))
processed_images_count = 0
total_potholes_found = 0

for label_path in all_label_files:
    with open(label_path, 'r') as file:
        lines = file.readlines()
        
    pothole_lines = []
    
    # 1. Read each bounding box line in the text file
    for line in lines:
        parts = line.strip().split()
        if not parts:
            continue
            
        class_id = parts[0]
        
        # 2. Keep the line ONLY if it is a pothole
        if class_id == OLD_POTHOLE_ID:
            # Change the old ID (4) to the new Single-Class ID (0)
            new_line = f"{NEW_POTHOLE_ID} " + " ".join(parts[1:]) + "\n"
            pothole_lines.append(new_line)
            total_potholes_found += 1
            
    # 3. If this image contains at least one pothole, save it to the new folder
    if len(pothole_lines) > 0:
        # Save the new filtered label
        new_label_path = target_labels_dir / label_path.name
        with open(new_label_path, 'w') as new_file:
            new_file.writelines(pothole_lines)
            
        # Copy the corresponding image (.jpg or .png)
        img_name_jpg = label_path.stem + '.jpg'
        img_name_png = label_path.stem + '.png'
        
        source_img_jpg = source_images_dir / img_name_jpg
        source_img_png = source_images_dir / img_name_png
        
        if source_img_jpg.exists():
            shutil.copy(str(source_img_jpg), str(target_images_dir / img_name_jpg))
            processed_images_count += 1
        elif source_img_png.exists():
            shutil.copy(str(source_img_png), str(target_images_dir / img_name_png))
            processed_images_count += 1

# ==========================================
# SUMMARY REPORT
# ==========================================
print("\n" + "="*40)
print("DATA EXTRACTION COMPLETE")
print("="*40)
print(f"[*] Total original text files scanned : {len(all_label_files)}")
print(f"[*] Total images copied (with potholes): {processed_images_count}")
print(f"[*] Total individual potholes bounded  : {total_potholes_found}")
print(f"[*] Clean dataset saved to            : {target_dir}")
print("="*40)

[*] Starting Data Extraction for N-RDD2024-MISS split...
[*] Filtering class ID '3' and converting to '0'...

DATA EXTRACTION COMPLETE
[*] Total original text files scanned : 6001
[*] Total images copied (with potholes): 884
[*] Total individual potholes bounded  : 1693
[*] Clean dataset saved to            : /home/sabda/code/pothole-detection/data/n-rdd2024_single_class
